In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

# ==========================================
# DEFINING PERFORMANCE METRICS FUNCTION
# ==========================================
def performance_measures(conf_mat):
    accuracy = np.trace(conf_mat)/np.sum(conf_mat)
    
    TN = conf_mat[0, 0]
    FN = conf_mat[0, 1]
    FP = conf_mat[1, 0]
    TP = conf_mat[1, 1]

    sensitivity = np.nan if (TP+FN)==0 else TP/(TP+FN)
    specificity = np.nan if (TN+FP)==0 else TN/(TN+FP)
    precision = np.nan if (TP+FP)== 0 else TP/(TP+FP)

    if np.isnan(precision) or np.isnan(sensitivity) or (precision+sensitivity)==0:
        f1_score=np.nan
    else:
        f1_score=2*precision*sensitivity/(precision+sensitivity)

    return {
        "accuracy": accuracy,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "F1_score": f1_score
    }

# ==========================================
# PERMUTATION IMPORTANCE FUNCTION
# ==========================================
def permutation_importance_accuracy(model, x_data, y_data, feature_names, random_state=42):
    rng = np.random.default_rng(random_state)
    baseline = np.mean(model.predict(x_data)==y_data)

    importance_rows=[]

    x_data=x_data.copy()

    for j, feature in enumerate(feature_names):
        xp=x_data.copy()
        xp[:,j]=rng.permutation(xp[:,j])

        perm_acc=np.mean(model.predict(xp)==y_data)

        importance_rows.append({
            "Variable": feature,
            "Importance": baseline-perm_acc
        })

    importance_df=pd.DataFrame(importance_rows).sort_values(
        by="Importance", ascending=False
    ).reset_index(drop=True)

    return importance_df


# ==========================================
# ROC PLOTTING FUNCTION
# ==========================================
def plot_roc_curve(y_true, pred_prob, title):
    fpr, tpr, _=roc_curve(y_true, pred_prob)
    auc = roc_auc_score(y_true, pred_prob)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, linewidth=2)
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    plt.title(title)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.grid(True)
    plt.show()

    return auc


pneumonia_data=pd.read_csv(
    r"C:/Users/000110888/OneDrive - CSULB/Desktop/pneumonia_data.csv")

# ==========================================
# ENCODING CATEGORICAL VARIABLES, MIN-MAX SCALING
# ==========================================
pneumonia_data["pneumonia"]=np.where(pneumonia_data["pneumonia"]=="yes", 1, 0)
pneumonia_data["gender"]=np.where(pneumonia_data["gender"]=="M", 1, 0)
pneumonia_data["tobacco_use"]=np.where(pneumonia_data["tobacco_use"]=="yes", 1, 0)

pneumonia_data["age"]=((pneumonia_data["age"]-pneumonia_data["age"].min())/
(pneumonia_data["age"].max()-pneumonia_data["age"].min()))

pneumonia_data["PM2_5"]=((pneumonia_data["PM2_5"]-pneumonia_data["PM2_5"].min())/
(pneumonia_data["PM2_5"].max()-pneumonia_data["PM2_5"].min()))

# ==========================================
# CREATING TRAINING AND TESTING SETS
# STRATIFYING BY PNEUMONIA
# ==========================================
X=pneumonia_data.drop(columns=["pneumonia"])
y=pneumonia_data["pneumonia"]

train_x_df, test_x_df, train_y, test_y = train_test_split(X, y, test_size=0.2,
random_state=447033, stratify=y)

train=pd.concat([train_x_df, train_y], axis=1)
test=pd.concat([test_x_df, test_y], axis=1)

# displaying target value distribution in training and testing sets
print("Training set target distribution:")
train_counts=train["pneumonia"].value_counts().sort_index()
train_pct=(train["pneumonia"].value_counts(normalize=True).sort_index()*100).round(2)
print(pd.DataFrame({"Count": train_counts, "Percentage": train_pct}))

print("\nTesting set target distribution:")
test_counts=test["pneumonia"].value_counts().sort_index()
test_pct=(test["pneumonia"].value_counts(normalize=True).sort_index()*100).round(2)
print(pd.DataFrame({"Count": test_counts, "Percentage": test_pct}))

# separating features and target in training and testing sets
train_x=train.drop(columns=["pneumonia"]).to_numpy()
train_y=train["pneumonia"].to_numpy()
test_x=test.drop(columns=["pneumonia"]).to_numpy()
test_y=test["pneumonia"].to_numpy()

feature_names=train.drop(columns=["pneumonia"]).columns.tolist()

###########################################
# FITTING RANDOM FOREST BINARY CLASSIFIER #
###########################################
rf_biclass=RandomForestClassifier(n_estimators=150, max_features=4, max_leaf_nodes=30,
random_state=330222)
rf_biclass.fit(train_x, train_y)

rf_imp_df=pd.DataFrame({
    "Variable": feature_names,
    "MeanDecreaseGini": rf_biclass.feature_importances_
}).sort_values(by="MeanDecreaseGini", ascending=False).reset_index(drop=True)

print("\nRandom Forest Binary Classifier - Feature Importance:")
print(rf_imp_df)

pred_class=rf_biclass.predict(test_x)

conf_mat=confusion_matrix(test_y, pred_class, labels=[0, 1]).T
print("\nRandom Forest Binary Classifier - Confusion Matrix:")
print(pd.DataFrame(conf_mat, index=["Predicted_0", "Predicted_1"], columns=["Actual_0", "Actual_1"]))

m=performance_measures(conf_mat)
print(f"RF Accuracy: {m['accuracy']:.4f}")
print(f"RF Sensitivity: {m['sensitivity']:.4f}")
print(f"RF Specificity: {m['specificity']:.4f}")
print(f"RF Precision: {m['precision']:.4f}")
print(f"RF F1-score: {m['F1_score']:.4f}")

pred_prob=rf_biclass.predict_proba(test_x)[:,1]
auc=plot_roc_curve(test_y, pred_prob, "ROC Curve for Random Forest Binary Classifier")
print(f"RF AUC: {auc:.4f}")

###############################################
# FITTING GRADIENT BOOSTING BINARY CLASSIFIER #
###############################################
xgb_biclass=XGBClassifier(objective="binary:logistic", n_estimators=300, max_depth=6,
learning_rate=0.01, subsample=0.8, colsample_bytree=0.5, random_state=558607,
eval_metric="logloss")
xgb_biclass.fit(train_x, train_y)

imp_df=pd.DataFrame({
    "Feature": feature_names,
    "Gain": xgb_biclass.feature_importances_
}).sort_values(by="Gain", ascending=False).reset_index(drop=True)

print("\nGradient Boosting Binary Classifier - Feature Importance:")
print(imp_df)

pred_prob=xgb_biclass.predict_proba(test_x)[:,1]
pred_class=(pred_prob >= 0.5).astype(int)

conf_mat=confusion_matrix(test_y, pred_class, labels=[0,1]).T
print("\nGradient Boosting Binary Classifier - Confusion Matrix:")
print(pd.DataFrame(conf_mat, index=["Predicted_0", "Predicted_1"], columns=["Actual_0", "Actual_1"]))

m=performance_measures(conf_mat)
print(f"XGBoost Accuracy: {m['accuracy']:.4f}")
print(f"XGBoost Sensitivity: {m['sensitivity']:.4f}")
print(f"XGBoost Specificity: {m['specificity']:.4f}")
print(f"XGBoost Precision: {m['precision']:.4f}")
print(f"XGBoost F1-score: {m['F1_score']:.4f}")

auc=plot_roc_curve(test_y, pred_prob, "ROC Curve for XGBoost Binary Classifier")
print(f"XGB AUC: {auc:.4f}")

###############################################################
# FITTING SUPPORT VECTOR BINARY CLASSIFIER WITH LINEAR KERNEL #
###############################################################
svm_class_linear=SVC(kernel="linear", probability=True, random_state=42)
svm_class_linear.fit(train_x, train_y)

importance=pd.DataFrame({
    "Variable": feature_names,
    "Importance": np.abs(svm_class_linear.coef_.ravel())
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

print("\nSVM (Linear Kernel) Binary Classifier - Feature Importance:")
print(importance)

pred_class=svm_class_linear.predict(test_x)

conf_mat=confusion_matrix(test_y, pred_class, labels=[0,1]).T
print("\nSVM (Linear Kernel) Binary Classifier - Confusion Matrix:")
print(pd.DataFrame(conf_mat, index=["Predicted_0", "Predicted_1"], columns=["Actual_0", "Actual_1"]))

m = performance_measures(conf_mat)
print(f"SVM (Linear Kernel) Accuracy: {m['accuracy']:.4f}")
print(f"SVM (Linear Kernel) Sensitivity: {m['sensitivity']:.4f}")
print(f"SVM (Linear Kernel) Specificity: {m['specificity']:.4f}")
print(f"SVM (Linear Kernel) Precision: {m['precision']:.4f}")
print(f"SVM (Linear Kernel) F1-score: {m['F1_score']:.4f}")

pred_prob=svm_class_linear.predict_proba(test_x)[:,1]
auc=plot_roc_curve(test_y, pred_prob, "ROC Curve for SVM (Linear Kernel) Binary Classifier")
print(f"SVM (Linear Kernel) AUC: {auc:.4f}")

###################################################################
# FITTING SUPPORT VECTOR BINARY CLASSIFIER WITH POLYNOMIAL KERNEL #
###################################################################
svm_class_poly=SVC(kernel="poly", probability=True, random_state=42)
svm_class_poly.fit(train_x, train_y)

importance=permutation_importance_accuracy(svm_class_poly, test_x, test_y, 
feature_names, random_state=42)

print("\nSVM (Polynomial Kernel) Binary Classifier - Feature Importance:")
print(importance)

pred_class=svm_class_poly.predict(test_x)

conf_mat=confusion_matrix(test_y, pred_class, labels=[0,1]).T
print("\nSVM (Polynomial Kernel) Binary Classifier - Confusion Matrix:")
print(pd.DataFrame(conf_mat, index=["Predicted_0", "Predicted_1"], columns=["Actual_0", "Actual_1"]))

m=performance_measures(conf_mat)
print(f"SVM (Polynomial Kernel) Accuracy: {m['accuracy']:.4f}")
print(f"SVM (Polynomial Kernel) Sensitivity: {m['sensitivity']:.4f}")
print(f"SVM (Polynomial Kernel) Specificity: {m['specificity']:.4f}")
print(f"SVM (Polynomial Kernel) Precision: {m['precision']:.4f}")
print(f"SVM (Polynomial Kernel) F1-score: {m['F1_score']:.4f}")

pred_prob=svm_class_poly.predict_proba(test_x)[:,1]
auc=plot_roc_curve(test_y, pred_prob, "ROC Curve for SVM (Polynomial Kernel) Binary Classifier")
print(f"SVM (Polynomial Kernel) AUC: {auc:.4f}")

###############################################################
# FITTING SUPPORT VECTOR BINARY CLASSIFIER WITH RADIAL KERNEL #
###############################################################
svm_class_radial=SVC(kernel="rbf", probability=True, random_state=42)
svm_class_radial.fit(train_x, train_y)

importance=permutation_importance_accuracy(svm_class_radial, test_x, test_y, 
feature_names, random_state=42)

print("\nSVM (Radial Kernel) Binary Classifier - Feature Importance:")
print(importance)

pred_class=svm_class_radial.predict(test_x)

conf_mat=confusion_matrix(test_y, pred_class, labels=[0,1]).T
print("\nSVM (Radial Kernel) Binary Classifier - Confusion Matrix:")
print(pd.DataFrame(conf_mat, index=["Predicted_0", "Predicted_1"], columns=["Actual_0", "Actual_1"]))

m=performance_measures(conf_mat)
print(f"SVM (Radial Kernel) Accuracy: {m['accuracy']:.4f}")
print(f"SVM (Radial Kernel) Sensitivity: {m['sensitivity']:.4f}")
print(f"SVM (Radial Kernel) Specificity: {m['specificity']:.4f}")
print(f"SVM (Radial Kernel) Precision: {m['precision']:.4f}")
print(f"SVM (Radial Kernel) F1-score: {m['F1_score']:.4f}")

pred_prob=svm_class_radial.predict_proba(test_x)[:,1]
auc=plot_roc_curve(test_y, pred_prob, "ROC Curve for SVM (Radial Kernel) Binary Classifier")
print(f"SVM (Radial Kernel) AUC: {auc:.4f}")

################################################################
# FITTING SUPPORT VECTOR BINARY CLASSIFIER WITH SIGMOID KERNEL #
################################################################
svm_class_sigmoid=SVC(kernel="sigmoid", probability=True, random_state=42)
svm_class_sigmoid.fit(train_x, train_y)

importance=permutation_importance_accuracy(svm_class_sigmoid, test_x, test_y, 
feature_names, random_state=42)

print("\nSVM (Sigmoid Kernel) Binary Classifier - Feature Importance:")
print(importance)

pred_class=svm_class_sigmoid.predict(test_x)

conf_mat=confusion_matrix(test_y, pred_class, labels=[0, 1]).T
print("\nSVM (Sigmoid Kernel) Binary Classifier - Confusion Matrix:")
print(pd.DataFrame(conf_mat, index=["Predicted_0", "Predicted_1"], columns=["Actual_0", "Actual_1"]))

m=performance_measures(conf_mat)
print(f"SVM (Sigmoid Kernel) Accuracy: {m['accuracy']:.4f}")
print(f"SVM (Sigmoid Kernel) Sensitivity: {m['sensitivity']:.4f}")
print(f"SVM (Sigmoid Kernel) Specificity: {m['specificity']:.4f}")
print(f"SVM (Sigmoid Kernel) Precision: {m['precision']:.4f}")
print(f"SVM (Sigmoid Kernel) F1-score: {m['F1_score']:.4f}")

pred_prob=svm_class_sigmoid.predict_proba(test_x)[:,1]
auc=plot_roc_curve(test_y, pred_prob, "ROC Curve for SVM (Sigmoid Kernel) Binary Classifier")
print(f"SVM (Sigmoid Kernel) AUC: {auc:.4f}")

################################################
# FITTING K-NEAREST NEIGHBOR BINARY CLASSIFIER #
################################################
knn_biclass=KNeighborsClassifier()
knn_biclass.fit(train_x, train_y)

importance=permutation_importance_accuracy(knn_biclass, test_x, test_y, 
feature_names, random_state=42)

print("\nKNN Binary Classifier - Feature Importance:")
print(importance)

pred_class=knn_biclass.predict(test_x)

conf_mat=confusion_matrix(test_y, pred_class, labels=[0, 1]).T
print("\nKNN Binary Classifier - Confusion Matrix:")
print(pd.DataFrame(conf_mat, index=["Predicted_0", "Predicted_1"], columns=["Actual_0", "Actual_1"]))

m=performance_measures(conf_mat)
print(f"KNN Accuracy: {m['accuracy']:.4f}")
print(f"KNN Sensitivity: {m['sensitivity']:.4f}")
print(f"KNN Specificity: {m['specificity']:.4f}")
print(f"KNN Precision: {m['precision']:.4f}")
print(f"KNN F1-score: {m['F1_score']:.4f}")

pred_prob=knn_biclass.predict_proba(test_x)[:,1]
auc=plot_roc_curve(test_y, pred_prob, "ROC Curve for KNN Binary Classifier")
print(f"KNN AUC: {auc:.4f}")

#########################################
# FITTING NAIVE BAYES BINARY CLASSIFIER #
#########################################
nb_biclass=GaussianNB()
nb_biclass.fit(train_x, train_y)

importance=permutation_importance_accuracy(nb_biclass, test_x, test_y, 
feature_names, random_state=42)

print("\nNaive Bayes Binary Classifier - Feature Importance:")
print(importance)

pred_class=nb_biclass.predict(test_x)

conf_mat=confusion_matrix(test_y, pred_class, labels=[0, 1]).T
print("\nNaive Bayes Binary Classifier - Confusion Matrix:")
print(pd.DataFrame(conf_mat, index=["Predicted_0", "Predicted_1"], columns=["Actual_0", "Actual_1"]))

m=performance_measures(conf_mat)
print(f"NB Accuracy: {m['accuracy']:.4f}")
print(f"NB Sensitivity: {m['sensitivity']:.4f}")
print(f"NB Specificity: {m['specificity']:.4f}")
print(f"NB Precision: {m['precision']:.4f}")
print(f"NB F1-score: {m['F1_score']:.4f}")

pred_prob=nb_biclass.predict_proba(test_x)[:,1]
auc=plot_roc_curve(test_y, pred_prob, "ROC Curve for Naive Bayes Binary Classifier")
print(f"NB AUC: {auc:.4f}")

#######################################################
# FITTING ARTIFICIAL NEURAL NETWORK BINARY CLASSIFIER #
#######################################################
ann_biclass=MLPClassifier(hidden_layer_sizes=(3,), activation="logistic",
solver="adam", max_iter=2000, random_state=345028)
ann_biclass.fit(train_x, train_y)

baseline_prob=ann_biclass.predict_proba(test_x)[:,1]

rng=np.random.default_rng(42)
perm_imp=[]

for j, feature in enumerate(feature_names):
    xp=test_x.copy()
    xp[:, j]=rng.permutation(xp[:, j])
    perm_prob=ann_biclass.predict_proba(xp)[:, 1]
    perm_imp.append(np.mean(np.abs(perm_prob - baseline_prob)))

importance=pd.DataFrame({
    "Variable": feature_names,
    "Importance": perm_imp
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

print("\nANN Binary Classifier - Feature Importance:")
print(importance)

pred_prob=ann_biclass.predict_proba(test_x)[:, 1]
pred_class=(pred_prob >= 0.5).astype(int)

conf_mat=confusion_matrix(test_y, pred_class, labels=[0, 1]).T
print("\nANN Binary Classifier - Confusion Matrix:")
print(pd.DataFrame(conf_mat, index=["Predicted_0", "Predicted_1"], columns=["Actual_0", "Actual_1"]))

m=performance_measures(conf_mat)
print(f"ANN Accuracy: {m['accuracy']:.4f}")
print(f"ANN Sensitivity: {m['sensitivity']:.4f}")
print(f"ANN Specificity: {m['specificity']:.4f}")
print(f"ANN Precision: {m['precision']:.4f}")
print(f"ANN F1-score: {m['F1_score']:.4f}")

auc=plot_roc_curve(test_y, pred_prob, "ROC Curve for ANN Model")
print(f"ANN AUC: {auc:.4f}")